# 🏛️ Nano ChessGPT — 85M Parameter Training

**Character-level chess language model trained on 26M Lichess Elite games.**

| Property | Value |
|----------|-------|
| Architecture | GPT-2 Small (12L × 12H × 768D) |
| Parameters | ~85 Million |
| Dataset | Lichess Elite DB (9.19B tokens) |
| Vocab | 29 chars (character-level) |
| Target | ~1500-1800 ELO |

---
### 📋 Session Workflow
1. **Cell 1** — GPU + Drive check  
2. **Cell 2** — Mount Google Drive  
3. **Cell 3** — Clone repo  
4. **Cell 4** — Download + Prepare dataset (~40 min, one-time)  
5. **Cell 5** — Start / Resume Training  
6. **Cell 6** — Monitor progress  

> ⚡ **After disconnect:** Run Cells 1→3, skip Cell 4 if data exists, run Cell 5 (auto-resumes from checkpoint)

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — GPU & Environment Check
# ════════════════════════════════════════════════════════
import torch, os, subprocess

print('=' * 55)
print(' Nano ChessGPT 85M — Environment Check')
print('=' * 55)

# GPU info
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'\n✅ GPU: {gpu}')
    print(f'   VRAM: {vram:.1f} GB')
    if vram < 14:
        print('   ⚠️  Low VRAM — reduce batch_size if OOM')
    else:
        print('   ✅ VRAM sufficient for training')
    print(f'   PyTorch: {torch.__version__}')
    print(f'   CUDA: {torch.version.cuda}')
    
    # Check bfloat16 support
    bf16 = torch.cuda.is_bf16_supported()
    print(f'   bfloat16: {"✅ supported" if bf16 else "❌ not supported (will use float16)"}')
else:
    print('❌ No GPU! Go to Runtime → Change runtime type → T4 GPU')

# Disk space
result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
lines = result.stdout.strip().split('\n')
disk_info = lines[1].split()
print(f'\n💾 Disk: {disk_info[3]} free of {disk_info[1]} total')

total_needed_gb = 6.5 + 16.3 + 0.9  # raw_zips + train.bin + val.bin
print(f'   Needed for dataset: ~{total_needed_gb:.1f} GB')

# RAM
import psutil
ram = psutil.virtual_memory()
print(f'   RAM: {ram.available/1024**3:.1f} GB free of {ram.total/1024**3:.1f} GB')
print()
print('✅ Ready for setup!')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — Mount Google Drive (for checkpoint saving)
# ════════════════════════════════════════════════════════
from google.colab import drive
import os

drive.mount('/content/drive')

# Create checkpoint directory in Drive
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
DRIVE_META_PATH      = '/content/drive/MyDrive/NanoChessGPT/meta.pkl'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

print(f'✅ Drive mounted!')
print(f'   Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}')

# Check if meta.pkl exists in Drive
if os.path.exists(DRIVE_META_PATH):
    import pickle
    meta = pickle.load(open(DRIVE_META_PATH, 'rb'))
    print(f'   ✅ meta.pkl found in Drive (vocab_size={meta["vocab_size"]})')
else:
    print(f'   ℹ️  meta.pkl not found in Drive — will be generated by prepare.py')

# Check for existing checkpoints (for resume)
ckpts = [f for f in os.listdir(DRIVE_CHECKPOINT_DIR) if f.endswith('.pt')] if os.path.exists(DRIVE_CHECKPOINT_DIR) else []
if ckpts:
    print(f'   🔄 Found {len(ckpts)} checkpoint(s) → training will RESUME!')
    for c in sorted(ckpts)[-3:]:
        print(f'      - {c}')
else:
    print(f'   🆕 No checkpoints found → fresh training will start')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — Clone Repo & Setup
# ════════════════════════════════════════════════════════
import os

REPO_DIR = '/content/NanoChessGPT-85M'
GITHUB_REPO = 'https://github.com/ItsMohammadSajid/NanoChessGPT-85M.git'

if os.path.exists(REPO_DIR):
    print(f'✅ Repo already exists at {REPO_DIR}')
    os.chdir(REPO_DIR)
else:
    print(f'📦 Cloning repo...')
    !git clone {GITHUB_REPO} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f'✅ Cloned to {REPO_DIR}')

print(f'📁 Working directory: {os.getcwd()}')
print(f'📋 Files:')
!ls -la

# Copy Drive checkpoints to local out dir (for resume)
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
LOCAL_OUT_DIR = f'{REPO_DIR}/out-chess-gpt2small'
os.makedirs(LOCAL_OUT_DIR, exist_ok=True)

ckpts = [f for f in os.listdir(DRIVE_CHECKPOINT_DIR) if f.endswith('.pt')] if os.path.exists(DRIVE_CHECKPOINT_DIR) else []
if ckpts:
    print(f'\n🔄 Copying {len(ckpts)} checkpoint(s) from Drive to local...')
    !cp {DRIVE_CHECKPOINT_DIR}/*.pt {LOCAL_OUT_DIR}/
    print(f'✅ Checkpoints ready for resume!')
else:
    print(f'\n🆕 No checkpoints to copy — fresh start')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — Download & Prepare Dataset
# ⚠️  SKIP THIS CELL if data/chess/train.bin already exists
# ⚠️  This takes ~40 min (download 6.4 GB + prepare 35 min)
# ════════════════════════════════════════════════════════
import os, time

REPO_DIR   = '/content/NanoChessGPT-85M'
CHESS_DIR  = f'{REPO_DIR}/data/chess'
TRAIN_BIN  = f'{CHESS_DIR}/train.bin'
DRIVE_META = '/content/drive/MyDrive/NanoChessGPT/meta.pkl'

os.chdir(CHESS_DIR)

# Check if already prepared
if os.path.exists(TRAIN_BIN):
    size_gb = os.path.getsize(TRAIN_BIN) / 1024**3
    print(f'✅ train.bin already exists ({size_gb:.1f} GB) — SKIPPING!')
    print(f'   Run Cell 5 to start training.')
else:
    print('=' * 55)
    print(' Step 1/2: Downloading Lichess Elite Database')
    print(' ~6.4 GB | ~5-10 min on Colab')
    print('=' * 55)
    t0 = time.time()
    !chmod +x download.sh && bash download.sh
    t1 = time.time()
    print(f'\n✅ Download complete in {(t1-t0)/60:.1f} min')
    
    print()
    print('=' * 55)
    print(' Step 2/2: Preparing Dataset (PGN → binary)')
    print(' ~35 min | Peak RAM: ~400 MB')
    print('=' * 55)
    t2 = time.time()
    !python prepare.py --input_dir=./raw_zips
    t3 = time.time()
    print(f'\n✅ Preparation complete in {(t3-t2)/60:.1f} min')
    
    # Copy meta.pkl to Drive
    if os.path.exists('meta.pkl'):
        !cp meta.pkl {DRIVE_META}
        print(f'✅ meta.pkl saved to Drive')
    
    # Final summary
    import pickle, numpy as np
    meta  = pickle.load(open('meta.pkl', 'rb'))
    train = np.memmap('train.bin', dtype=np.uint16, mode='r')
    val   = np.memmap('val.bin',   dtype=np.uint16, mode='r')
    total = len(train) + len(val)
    print()
    print('=' * 55)
    print(f'  vocab_size  : {meta["vocab_size"]}')
    print(f'  train tokens: {len(train):,}')
    print(f'  val tokens  : {len(val):,}')
    print(f'  total tokens: {total:,} ({total/1e9:.2f}B)')
    print(f'  STATUS: READY FOR TRAINING! ✅')
    print('=' * 55)
    
    # Clean up raw_zips to save disk
    print('\n🗑️  Removing raw_zips to free disk space...')
    !rm -rf raw_zips/
    print(f'✅ Disk freed! Run Cell 5 to start training.')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — Configure Checkpointing to Drive & Train
# ════════════════════════════════════════════════════════
import os, shutil

REPO_DIR              = '/content/NanoChessGPT-85M'
LOCAL_OUT_DIR         = f'{REPO_DIR}/out-chess-gpt2small'
DRIVE_CHECKPOINT_DIR  = '/content/drive/MyDrive/NanoChessGPT/checkpoints'

os.chdir(REPO_DIR)
os.makedirs(LOCAL_OUT_DIR, exist_ok=True)

# Check for existing checkpoint → auto detect resume
existing_ckpts = [f for f in os.listdir(LOCAL_OUT_DIR) if f.endswith('.pt')]
if existing_ckpts:
    INIT_FROM = 'resume'
    print(f'🔄 RESUMING from checkpoint: {sorted(existing_ckpts)[-1]}')
else:
    INIT_FROM = 'scratch'
    print(f'🆕 Starting FRESH training')

# Verify dataset exists
TRAIN_BIN = f'{REPO_DIR}/data/chess/train.bin'
if not os.path.exists(TRAIN_BIN):
    print('❌ train.bin not found! Run Cell 4 first.')
    raise FileNotFoundError('Run Cell 4 to prepare dataset!')
else:
    size_gb = os.path.getsize(TRAIN_BIN) / 1024**3
    print(f'✅ Dataset: train.bin ({size_gb:.1f} GB)')

print(f'\n🚀 Starting training...')
print(f'   Config: config/train_chess_gpt2small.py')
print(f'   Mode: {INIT_FROM}')
print(f'   Checkpoints: {LOCAL_OUT_DIR}')
print(f'   Drive backup: {DRIVE_CHECKPOINT_DIR}')
print()

!python train.py config/train_chess_gpt2small.py \
    --init_from={INIT_FROM} \
    --out_dir=out-chess-gpt2small

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — Save Checkpoint to Drive (run anytime)
# Run this BEFORE your session expires!
# ════════════════════════════════════════════════════════
import os, shutil, glob

LOCAL_OUT_DIR        = '/content/NanoChessGPT-85M/out-chess-gpt2small'
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/NanoChessGPT/checkpoints'

os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

ckpt_files = glob.glob(f'{LOCAL_OUT_DIR}/*.pt')

if not ckpt_files:
    print('❌ No checkpoint files found!')
    print(f'   Looking in: {LOCAL_OUT_DIR}')
    print(f'   Files: {os.listdir(LOCAL_OUT_DIR) if os.path.exists(LOCAL_OUT_DIR) else "dir not found"}')
else:
    print(f'💾 Saving {len(ckpt_files)} checkpoint(s) to Drive...')
    for f in ckpt_files:
        fname = os.path.basename(f)
        dst = f'{DRIVE_CHECKPOINT_DIR}/{fname}'
        shutil.copy2(f, dst)
        size_mb = os.path.getsize(dst) / 1024**2
        print(f'   ✅ {fname} → Drive ({size_mb:.0f} MB)')
    
    print(f'\n✅ All checkpoints saved to Drive!')
    print(f'   Location: {DRIVE_CHECKPOINT_DIR}')
    print(f'   Next session: Run Cells 1→3 then Cell 5 (auto-resumes)')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — Sample / Test the Model (after training)
# ════════════════════════════════════════════════════════
import os
os.chdir('/content/NanoChessGPT-85M')

# Generate chess moves
!python sample.py \
    --out_dir=out-chess-gpt2small \
    --start="e4 e5 Nf3 " \
    --num_samples=5 \
    --max_new_tokens=200 \
    --temperature=0.8 \
    --top_k=10

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — Training Progress Monitor
# ════════════════════════════════════════════════════════
import re, glob, os

LOCAL_OUT_DIR = '/content/NanoChessGPT-85M/out-chess-gpt2small'

# Check checkpoint info
ckpt_files = glob.glob(f'{LOCAL_OUT_DIR}/*.pt')
if ckpt_files:
    import torch
    for ckpt_path in sorted(ckpt_files):
        try:
            ckpt = torch.load(ckpt_path, map_location='cpu')
            iter_num   = ckpt.get('iter_num', 'unknown')
            best_val   = ckpt.get('best_val_loss', 'unknown')
            config     = ckpt.get('config', {})
            max_iters  = config.get('max_iters', 600000)
            
            progress = iter_num / max_iters * 100 if isinstance(iter_num, int) else 0
            
            print(f'\n📊 Checkpoint: {os.path.basename(ckpt_path)}')
            print(f'   Iteration:  {iter_num:,} / {max_iters:,} ({progress:.1f}%)')
            print(f'   Best val loss: {best_val:.4f}' if isinstance(best_val, float) else f'   Best val loss: {best_val}')
            
            # Estimate remaining time
            if isinstance(iter_num, int) and iter_num > 0:
                remaining = max_iters - iter_num
                # ~1.2 sec/iter on T4
                est_hours = remaining * 1.2 / 3600
                print(f'   Remaining:  ~{est_hours:.1f} hours (est. at 1.2s/iter on T4)')
        except Exception as e:
            print(f'   Error reading {ckpt_path}: {e}')
else:
    print('No checkpoints found yet. Training needs to run first.')